In [1]:
# conda activate scanpy
# Import necessary libraries
import scanpy as sc
import anndata
import os
import re  # For regular expression operations

In [ ]:
text = sc.read_loom("/storage/liuxiaodongLab/jiangjing/Projects/YutingFu/PD_YutingFu/scvelo/201129A_P1A/velocyto/201129A_P1A.loom")

In [21]:
# Define the list of sample names in order from sample1 to sample9
sample_names = [f'{i}' for i in ("P1A","P1B")]
print("Sample names to process:", sample_names)

Sample names to process: ['P1A', 'P1B']


In [24]:
# Initialize an empty list to store AnnData objects
adata_list = []

In [25]:
# Loop through each sample to read the loom file and adjust barcodes
for sample_name in sample_names:
    loom_file_path = f'/storage/liuxiaodongLab/jiangjing/Projects/YutingFu/PD_YutingFu/scvelo/201129A_{sample_name}/velocyto/201129A_{sample_name}.loom'
    # Check if the loom file exists
    if not os.path.isfile(loom_file_path):
        print(f"Warning: Loom file not found for {sample_name}. Skipping.")
        continue
    
    # Read the loom file into an AnnData object
    adata = sc.read_loom(loom_file_path)
    
    # 直接定义新前缀（例如 sample_name = 'patient1' → 前缀为 'patient1_'）
    prefix = f"{sample_name}_"

    # 定义转换函数：删除前缀（如 '201129A_P1_A:'）和末尾的 'x'，并添加新前缀
    def transform_barcode(barcode):
        # 删除冒号前的所有内容和末尾的 'x'
        cleaned = re.sub(r'^.*:', '', barcode)  # 删除前缀（如 '201129A_P1_A:'）
        cleaned = re.sub(r'x$', '', cleaned)    # 删除末尾的 'x'
        return f"{prefix}{cleaned}-1"             # 拼接新格式（如 'P1_A_AAACGCTTCTCGGTCT'）
    
    # 打印原始 barcode 的前3个
    print(f"Original barcodes for {sample_name}: {adata.obs.index[:3].tolist()}")

    # 应用转换函数到所有 barcode，更新索引（无需 sample_num 参数）
    adata.obs.index = adata.obs.index.map(transform_barcode)

    # 确保基因名唯一（保留此步骤，合并时仍需要）
    adata.var_names_make_unique()

    # 将处理后的 adata 加入列表
    adata_list.append(adata)

    # 打印处理后的统计信息
    print(f"Processed {sample_name}: {adata.n_obs} cells, {adata.n_vars} genes")
    print(f"First few transformed barcodes for {sample_name}: {adata.obs.index[:3].tolist()}")

/home/liuxiaodongLab/jiangjing/.local/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Original barcodes for P1A: [np.str_('201129A_P1_A:AAACGCTTCTCGGTCTx'), np.str_('201129A_P1_A:AAACCCAAGACAAGCCx'), np.str_('201129A_P1_A:AAACGAACACATCCCTx')]
Processed P1A: 22660 cells, 38606 genes
First few transformed barcodes for P1A: ['P1A_AAACGCTTCTCGGTCT-1', 'P1A_AAACCCAAGACAAGCC-1', 'P1A_AAACGAACACATCCCT-1']
Original barcodes for P1B: [np.str_('201129A_P1_B:AAACGAATCGGTCGACx'), np.str_('201129A_P1_B:AAACCCACACCTATCCx'), np.str_('201129A_P1_B:AAACGCTAGGATTTGAx')]
Processed P1B: 19300 cells, 38606 genes
First few transformed barcodes for P1B: ['P1B_AAACGAATCGGTCGAC-1', 'P1B_AAACCCACACCTATCC-1', 'P1B_AAACGCTAGGATTTGA-1']


/home/liuxiaodongLab/jiangjing/.local/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [26]:
# Merge all AnnData objects into one
# Use join='outer' to include all genes, label='batch' to track sample origin
adata_merged = anndata.concat(adata_list, axis=0, join='outer', label='batch', keys=sample_names)
print(f"Merged AnnData: {adata_merged.n_obs} cells, {adata_merged.n_vars} genes")

Merged AnnData: 41960 cells, 38606 genes


In [27]:
# Verify that all barcodes are unique after merging
print("Number of cells:", len(adata_merged.obs_names))
print("Number of unique barcodes:", len(set(adata_merged.obs_names)))
if len(adata_merged.obs_names) == len(set(adata_merged.obs_names)):
    print("All barcodes are unique.")
else:
    print("Warning: There are duplicate barcodes.")

Number of cells: 41960
Number of unique barcodes: 41960
All barcodes are unique.


In [28]:
# Display the first few barcodes to check the suffix renaming
print("First few barcodes:")
print(adata_merged.obs_names[:5])

First few barcodes:
Index(['P1A_AAACGCTTCTCGGTCT-1', 'P1A_AAACCCAAGACAAGCC-1',
       'P1A_AAACGAACACATCCCT-1', 'P1A_AAACGCTCACTCGATA-1',
       'P1A_AAACGCTTCGGTCTAA-1'],
      dtype='object', name='CellID')


In [29]:
# Save the merged AnnData object as a loom file
merged_loom_path = '/storage/liuxiaodongLab/jiangjing/Projects/YutingFu/PD_YutingFu/scvelo/merged_P1A_P1B.loom'
adata_merged.write_loom(merged_loom_path)
print(f"Merged loom file saved to: {merged_loom_path}")

Merged loom file saved to: /storage/liuxiaodongLab/jiangjing/Projects/YutingFu/PD_YutingFu/scvelo/merged_P1A_P1B.loom


In [ ]:
adata = sc.read_loom("merged_P1A_P1B.loom")